# Chapter 1: Databases With Python — MySQL & MongoDB

### Why This Matters for a Data Analyst

Excel and CSV files become unmanageable beyond a few hundred thousand rows, and — more importantly — they're static snapshots. A database gives you **live, consistent, multi-user** data: write a query once, and you always get the current numbers, not whatever was true when someone last exported a spreadsheet. Most companies split their data across two broad database styles, and you'll likely touch both over a career: **relational** databases (MySQL, PostgreSQL, SQL Server) for structured transactional data with fixed schemas, and **document** databases (MongoDB) for flexible, hierarchical data whose shape varies record to record.

### 1.1 MySQL — Relational Data

In [ ]:
import mysql.connector
from mysql.connector import Error
import pandas as pd

def get_mysql_connection():
    try:
        conn = mysql.connector.connect(
            host="localhost",
            user="root",
            password="root",           # never hardcode real credentials — use environment variables in practice
            database="business_db"
        )
        if conn.is_connected():
            print("Connected to MySQL successfully!")
            return conn
    except Error as e:
        print("Error while connecting:", e)
        return None

⚠️ The `password="root"` shown here is a placeholder for learning purposes only. In any real project, credentials belong in environment variables (`os.environ["DB_PASSWORD"]`) or a secrets manager, never typed directly into a script that might end up in version control.

**The SQL clauses that cover the overwhelming majority of an analyst's actual querying needs:**

| Clause | Purpose | Example |
|---|---|---|
| `WHERE` | Filter rows | `WHERE region = 'West'` |
| `GROUP BY` | Aggregate | `GROUP BY product` |
| `HAVING` | Filter *after* aggregation | `HAVING SUM(qty) > 100` |
| `ORDER BY` | Sort | `ORDER BY revenue DESC` |
| `LIMIT` | Top-N | `LIMIT 20` |
| `JOIN` | Combine tables | `INNER JOIN customers ON ...` |

```sql
-- Total revenue by region
SELECT region, SUM(quantity * unit_price) AS revenue
FROM sales
GROUP BY region
ORDER BY revenue DESC;

-- Combining two tables — exactly the SQL equivalent of Volume 02's pd.merge
SELECT s.order_date, c.customer_name, s.product, s.quantity * s.unit_price AS amount
FROM sales s
INNER JOIN customers c ON s.customer_id = c.id
WHERE s.order_date >= '2025-01-01';
```

💡 **`WHERE` vs. `HAVING` is a genuinely common point of confusion worth being precise about:** `WHERE` filters individual rows *before* any grouping/aggregation happens; `HAVING` filters *groups*, after aggregation, based on an aggregate condition. "Show me sales over ₹1,000" (a per-row condition) is `WHERE`; "show me products whose *total* sales exceed ₹1,000,000" (a per-group condition) requires `HAVING`, since the total doesn't exist until after grouping.

**Bringing query results straight into Pandas** — the bridge back into everything from Volumes 02–09:

In [ ]:
def query_to_df(conn, sql):
    cursor = conn.cursor()
    cursor.execute(sql)
    columns = [col[0] for col in cursor.description]
    data = cursor.fetchall()
    cursor.close()
    return pd.DataFrame(data, columns=columns)

# conn = get_mysql_connection()
# sql = "SELECT region, product, SUM(quantity*unit_price) AS revenue FROM sales GROUP BY region, product"
# df = query_to_df(conn, sql)
# conn.close()

💡 **The professional discipline worth adopting from day one:** always filter and aggregate as much as possible *in the SQL query itself* (`WHERE`, `GROUP BY`) rather than pulling an entire raw table into Pandas and filtering afterward. A database engine is specifically optimized to filter and aggregate large tables efficiently; pulling millions of unneeded rows across the network into memory just to immediately discard most of them in Pandas wastes both time and memory for no benefit.

### 1.2 MongoDB — Document Data

In [ ]:
from pymongo import MongoClient

# client = MongoClient("mongodb://localhost:27017/")
# db = client["business_db"]
# collection = db["customers"]

doc = {
    "customer_id": 1001,
    "name": "Priya Sharma",
    "city": "Mumbai",
    "segment": "Premium",
    "orders": [                                       # an ARRAY nested directly inside the document
        {"date": "2025-03-12", "amount": 4500},
        {"date": "2025-06-01", "amount": 8200}
    ]
}
# collection.insert_one(doc)

The defining difference from MySQL: this single customer document contains its *entire order history nested inside it*, as a list of sub-documents — something a relational table would instead require a separate `orders` table joined by `customer_id` to represent. This flexibility is exactly why MongoDB suits data whose shape varies or nests naturally: customer profiles with variable fields, event/clickstream logs, product catalogs where different product categories have genuinely different attributes.

In [ ]:
# Basic queries
# list(collection.find({"city": "Mumbai"}))
# list(collection.find({"segment": "Premium"}).limit(10))

# Aggregation pipeline — MongoDB's answer to SQL's GROUP BY, expressed as a sequence of stages
pipeline = [
    {"$match": {"segment": "Premium"}},                              # like SQL's WHERE
    {"$group": {
        "_id": "$city",
        "total_customers": {"$sum": 1},
        "avg_orders": {"$avg": {"$size": "$orders"}}
    }},                                                                # like SQL's GROUP BY + aggregate functions
    {"$sort": {"total_customers": -1}}                                 # like SQL's ORDER BY
]
# results = list(collection.aggregate(pipeline))

MongoDB's **aggregation pipeline** — a list of sequential `$match`/`$group`/`$sort` stages, each transforming the data before passing it to the next stage — is conceptually the exact same "filter, then aggregate, then sort" logic as SQL, just expressed as a Python list of dictionaries instead of a SQL string. If you're comfortable with `GROUP BY` from Section 1.1, this pipeline shape will feel familiar once you map `$match` → `WHERE`, `$group` → `GROUP BY`, `$sort` → `ORDER BY`.

### Cheat Sheet — Databases

| Task | MySQL (SQL) | MongoDB (Python) |
|---|---|---|
| Filter rows | `WHERE col = x` | `collection.find({"col": x})` |
| Aggregate | `GROUP BY col` | `{"$group": {"_id": "$col", ...}}` |
| Filter after aggregation | `HAVING SUM(x) > y` | Add a `$match` stage after `$group` |
| Sort | `ORDER BY col DESC` | `{"$sort": {"col": -1}}` |
| Limit | `LIMIT n` | `.limit(n)` |
| Combine tables | `JOIN` | `$lookup` (aggregation stage) |
| Load into Pandas | `pd.DataFrame(cursor.fetchall(), columns=...)` | `pd.DataFrame(list(collection.find(...)))` |

---